# 2. Data Cleaning and Feature Engineering

This notebook prepares the UCI Default of Credit Card Clients dataset for machine learning.

The main objectives are:

- Load and verify the raw dataset
- Clean invalid or undocumented categorical values
- Preserve meaningful payment-status information
- Identify and handle problematic data values
- Create meaningful credit-risk features
- Separate features and target
- Split the data into training and testing sets
- Prepare preprocessing steps for machine learning

The raw dataset is kept unchanged to preserve data provenance and reproducibility.

In [32]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

In [34]:
file_path = "../data/raw/default of credit card clients.xls"

df = pd.read_excel(file_path, header=1)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (30000, 25)


In [35]:
target = "default payment next month"

print("Target variable:", target)
print(df[target].value_counts())


Target variable: default payment next month
default payment next month
0    23364
1     6636
Name: count, dtype: int64


Clean EDUCATION

In [36]:
# For this dataset, the documented meaningful groups are:
# 1 = Graduate school
# 2 = University
# 3 = High school
# 4 = Others

# So we will combine 0, 5, and 6 into 4.

df["EDUCATION"] = df["EDUCATION"].replace({
    0: 4,
    5: 4,
    6: 4
})

In [37]:
# Verify the changes made to the EDUCATION column

print(df["EDUCATION"].value_counts().sort_index()) 

EDUCATION
1    10585
2    14030
3     4917
4      468
Name: count, dtype: int64


Clean MARRIAGE

In [38]:
# Your EDA showed:
# 0, 1, 2, 3

# We will treat 0 as the "Others" category:

df["MARRIAGE"] = df["MARRIAGE"].replace({
    0: 3
})

In [39]:
# Verify

print(df["MARRIAGE"].value_counts().sort_index())

MARRIAGE
1    13659
2    15964
3      377
Name: count, dtype: int64


Define feature groups

In [40]:
pay_cols = [
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

bill_cols = [
    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6"
]

pay_amt_cols = [
    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6"
]

Create payment behavior features

In [41]:
# Average payment delay

df["AVG_PAYMENT_DELAY"] = df[pay_cols].mean(axis=1)

# Maximum payment delay

df["MAX_PAYMENT_DELAY"] = df[pay_cols].max(axis=1)

# Number of delayed months

df["DELAYED_MONTHS"] = (df[pay_cols] > 0).sum(axis=1)

#Average bill amount

df["AVG_BILL_AMT"] = df[bill_cols].mean(axis=1)

#Maximum bill amount

df["MAX_BILL_AMT"] = df[bill_cols].max(axis=1)

#Total bill amount

df["TOTAL_BILL_AMT"] = df[bill_cols].sum(axis=1)

#Average previous payment

df["AVG_PAY_AMT"] = df[pay_amt_cols].mean(axis=1)

#Total previous payments

df["TOTAL_PAY_AMT"] = df[pay_amt_cols].sum(axis=1)

Create credit utilization features

In [42]:
for bill_col in bill_cols:
    month = bill_col.replace("BILL_AMT", "")
    
    df[f"UTILIZATION_{month}"] = (
        df[bill_col] / df["LIMIT_BAL"]
    )

In [43]:
util_cols = [
    "UTILIZATION_1",
    "UTILIZATION_2",
    "UTILIZATION_3",
    "UTILIZATION_4",
    "UTILIZATION_5",
    "UTILIZATION_6"
]

In [44]:
df["AVG_UTILIZATION"] = df[util_cols].mean(axis=1)

df["MAX_UTILIZATION"] = df[util_cols].max(axis=1)

Check the engineered features

In [45]:
engineered_cols = [
    "AVG_PAYMENT_DELAY",
    "MAX_PAYMENT_DELAY",
    "DELAYED_MONTHS",
    "AVG_BILL_AMT",
    "MAX_BILL_AMT",
    "TOTAL_BILL_AMT",
    "AVG_PAY_AMT",
    "TOTAL_PAY_AMT",
    "AVG_UTILIZATION",
    "MAX_UTILIZATION"
]

df[engineered_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
AVG_PAYMENT_DELAY,30000.0,-0.182439,0.982176,-2.000000,-0.833333,0.000000,0.000000,6.000000e+00
MAX_PAYMENT_DELAY,30000.0,0.438733,1.345154,-2.000000,0.000000,0.000000,2.000000,8.000000e+00
DELAYED_MONTHS,30000.0,0.834200,1.554303,0.000000,0.000000,0.000000,1.000000,6.000000e+00
AVG_BILL_AMT,30000.0,44976.945200,63260.721860,-56043.166667,4781.333333,21051.833333,57104.416667,8.773138e+05
MAX_BILL_AMT,30000.0,60572.436867,78404.814025,-6029.000000,10060.000000,31208.500000,79599.000000,1.664089e+06
TOTAL_BILL_AMT,30000.0,269861.671200,379564.331162,-336259.000000,28688.000000,126311.000000,342626.500000,5.263883e+06
AVG_PAY_AMT,30000.0,5275.232094,10137.946323,0.000000,1113.291667,2397.166667,5583.916667,6.273443e+05
TOTAL_PAY_AMT,30000.0,31651.392567,60827.677939,0.000000,6679.750000,14383.000000,33503.500000,3.764066e+06
AVG_UTILIZATION,30000.0,0.373048,0.351890,-0.232590,0.029997,0.284834,0.687929,5.364308e+00
MAX_UTILIZATION,30000.0,0.494965,0.433046,-0.100000,0.070636,0.430702,0.923246,1.068858e+01


In [46]:
df[engineered_cols].head()

,AVG_PAYMENT_DELAY,MAX_PAYMENT_DELAY,DELAYED_MONTHS,AVG_BILL_AMT,MAX_BILL_AMT,TOTAL_BILL_AMT,AVG_PAY_AMT,TOTAL_PAY_AMT,AVG_UTILIZATION,MAX_UTILIZATION
0,-0.333333,2,2,1284.000000,3913,7704,114.833333,689,0.064200,0.195650
1,0.500000,2,2,2846.166667,3455,17077,833.333333,5000,0.023718,0.028792
2,0.000000,0,0,16942.166667,29239,101653,1836.333333,11018,0.188246,0.324878
3,0.000000,0,0,38555.666667,49291,231334,1398.000000,8388,0.771113,0.985820
4,-0.333333,0,0,18223.166667,35835,109339,9841.500000,59049,0.364463,0.716700


Check for infinite values

In [47]:
print(
    "Infinite values:",
    np.isinf(df[engineered_cols]).sum().sum()
)

print(
    "NaN values:",
    df[engineered_cols].isnull().sum().sum()
)

Infinite values: 0
NaN values: 0


Check the new dataset

In [48]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 30000
Columns: 41


Remove ID from model features

In [49]:
X = df.drop(columns=[target, "ID"])
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (30000, 39)
y shape: (30000,)


In [50]:
# Check feature list

print("Number of features:", X.shape[1])

print("\nFeatures:")
for feature in X.columns:
    print(feature)

Number of features: 39

Features:
LIMIT_BAL
SEX
EDUCATION
MARRIAGE
AGE
PAY_0
PAY_2
PAY_3
PAY_4
PAY_5
PAY_6
BILL_AMT1
BILL_AMT2
BILL_AMT3
BILL_AMT4
BILL_AMT5
BILL_AMT6
PAY_AMT1
PAY_AMT2
PAY_AMT3
PAY_AMT4
PAY_AMT5
PAY_AMT6
AVG_PAYMENT_DELAY
MAX_PAYMENT_DELAY
DELAYED_MONTHS
AVG_BILL_AMT
MAX_BILL_AMT
TOTAL_BILL_AMT
AVG_PAY_AMT
TOTAL_PAY_AMT
UTILIZATION_1
UTILIZATION_2
UTILIZATION_3
UTILIZATION_4
UTILIZATION_5
UTILIZATION_6
AVG_UTILIZATION
MAX_UTILIZATION


Train/Test Split

In [53]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (24000, 39)
Testing set: (6000, 39)


In [54]:
# Verify target proportions

print("Overall default rate:")
print(y.mean())

print("\nTraining default rate:")
print(y_train.mean())

print("\nTesting default rate:")
print(y_test.mean())

Overall default rate:
0.2212

Training default rate:
0.22120833333333334

Testing default rate:
0.22116666666666668


In [ ]:
import os

# Create the output directory if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Save the prepared datasets
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Datasets saved successfully in ../data/processed/")

Datasets saved successfully in ../data/processed/
